<a href="https://colab.research.google.com/github/M-Santhoshkumar-2005/ITA0616-MACHINE-LEARNING-SLOT-A-/blob/main/Experiment_19.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd


dataset = [
    ["Graduate", "Salaried", 25000, 1, 760, 0],
    ["Not_Graduate", "Self_Employed", 22000, 0, 640, 0],
    ["Graduate", "Salaried", 30000, 1, 780, 1],
    ["Not_Graduate", "Salaried", 28000, 0, 670, 0],
    ["Graduate", "Self_Employed", 35000, 1, 740, 1],
    ["Graduate", "Salaried", 48000, 1, 810, 1],
    ["Not_Graduate", "Self_Employed", 26000, 0, 650, 0],
    ["Graduate", "Salaried", 55000, 1, 830, 1],
    ["Not_Graduate", "Salaried", 32000, 1, 720, 1],
    ["Graduate", "Self_Employed", 24000, 0, 630, 0],
    ["Graduate", "Salaried", 62000, 1, 850, 1],
    ["Not_Graduate", "Self_Employed", 40000, 1, 730, 1],
    ["Graduate", "Salaried", 27000, 0, 660, 0],
    ["Graduate", "Self_Employed", 52000, 1, 790, 1],
    ["Not_Graduate", "Salaried", 29000, 1, 710, 1],
    ["Graduate", "Salaried", 70000, 1, 860, 1],
]

df = pd.DataFrame(
    dataset,
    columns=["Education", "Employment", "Annual_Income", "Credit_History", "CIBIL_Score", "Loan_Approved"],
)

df_proc = df.copy()
df_proc["Income_Tier"] = pd.cut(
    df_proc["Annual_Income"],
    bins=[0, 35000, 55000, 200000],
    labels=["Low", "Medium", "High"],
)
df_proc["CIBIL_Tier"] = pd.cut(
    df_proc["CIBIL_Score"],
    bins=[0, 680, 760, 1000],
    labels=["Poor", "Fair", "Good"],
)
df_proc["Credit_Hist_Cat"] = df_proc["Credit_History"].map({0: "No", 1: "Yes"})

cat_cols = ["Education", "Employment", "Income_Tier", "Credit_Hist_Cat", "CIBIL_Tier"]
target_col = "Loan_Approved"

test_idx = [3, 6, 9, 12, 14]
train_mask = np.ones(len(df_proc), dtype=bool)
train_mask[test_idx] = False

train_df = df_proc[train_mask].reset_index(drop=True)
test_df = df_proc.iloc[test_idx].reset_index(drop=True)

alpha = 1.0
classes = sorted(train_df[target_col].unique())
priors = {}
cond_probs = {}
domains = {col: sorted(train_df[col].astype(str).unique()) for col in cat_cols}

for c in classes:
    sub = train_df[train_df[target_col] == c]
    n_c = len(sub)
    priors[c] = n_c / len(train_df)
    cond_probs[c] = {}
    for col in cat_cols:
        vals = sub[col].astype(str)
        v_len = len(domains[col])
        counts = vals.value_counts().to_dict()
        cond_probs[c][col] = {
            v: (counts.get(v, 0) + alpha) / (n_c + alpha * v_len) for v in domains[col]
        }


def predict_row(row):
    scores = {}
    for c in classes:
        prob = priors[c]
        for col in cat_cols:
            v = str(row[col])
            v_len = len(domains[col])
            n_c = int(round(priors[c] * len(train_df)))
            prob *= cond_probs[c][col].get(v, alpha / (max(n_c, 1) + alpha * v_len))
        scores[c] = prob
    total = sum(scores.values())
    if total > 0:
        for c in scores:
            scores[c] /= total
    best = max(scores, key=scores.get)
    return best, scores


preds = []
probs_list = []
for _, r in test_df.iterrows():
    p_cls, p_map = predict_row(r)
    preds.append(p_cls)
    probs_list.append(p_map)

preds = np.array(preds, dtype=int)
y_test = test_df[target_col].values.astype(int)
accuracy = np.mean(preds == y_test) * 100

label_names = {0: "Rejected", 1: "Approved"}
print(f"Test Accuracy: {accuracy:.2f}%\n")

for idx, actual, pred, p_map in zip(test_idx, y_test, preds, probs_list):
    raw = df.iloc[idx]
    p_fmt = {label_names[k]: round(v, 4) for k, v in p_map.items()}
    print(
        f"Row {idx + 1:2d} | {raw['Education']:12s}, {raw['Employment']:13s}, "
        f"Inc=₹{raw['Annual_Income']:5d}, Hist={raw['Credit_History']}, CIBIL={raw['CIBIL_Score']} | "
        f"Actual={label_names[actual]:8s} | Pred={label_names[pred]:8s} | Probs={p_fmt}"
    )

new_applicants = pd.DataFrame([
    ["Graduate", "Salaried", 50000, 1, 800],
    ["Not_Graduate", "Self_Employed", 24000, 0, 645],
], columns=["Education", "Employment", "Annual_Income", "Credit_History", "CIBIL_Score"])

new_applicants["Income_Tier"] = pd.cut(
    new_applicants["Annual_Income"],
    bins=[0, 35000, 55000, 200000],
    labels=["Low", "Medium", "High"],
)
new_applicants["CIBIL_Tier"] = pd.cut(
    new_applicants["CIBIL_Score"],
    bins=[0, 680, 760, 1000],
    labels=["Poor", "Fair", "Good"],
)
new_applicants["Credit_Hist_Cat"] = new_applicants["Credit_History"].map({0: "No", 1: "Yes"})

print("\nNew Loan Predictions:")
for i, r in new_applicants.iterrows():
    p_cls, p_map = predict_row(r)
    p_fmt = {label_names[k]: round(v, 4) for k, v in p_map.items()}
    print(
        f"Applicant {i + 1} | {r['Education']:12s}, {r['Employment']:13s}, "
        f"Inc=₹{r['Annual_Income']:5d}, Hist={r['Credit_History']}, CIBIL={r['CIBIL_Score']} | "
        f"Pred={label_names[p_cls]:8s} | Probs={p_fmt}"
    )

Test Accuracy: 100.00%

Row  4 | Not_Graduate, Salaried     , Inc=₹28000, Hist=0, CIBIL=670 | Actual=Rejected | Pred=Rejected | Probs={'Rejected': 0.9383, 'Approved': 0.0617}
Row  7 | Not_Graduate, Self_Employed, Inc=₹26000, Hist=0, CIBIL=650 | Actual=Rejected | Pred=Rejected | Probs={'Rejected': 0.9638, 'Approved': 0.0362}
Row 10 | Graduate    , Self_Employed, Inc=₹24000, Hist=0, CIBIL=630 | Actual=Rejected | Pred=Rejected | Probs={'Rejected': 0.9089, 'Approved': 0.0911}
Row 13 | Graduate    , Salaried     , Inc=₹27000, Hist=0, CIBIL=660 | Actual=Rejected | Pred=Rejected | Probs={'Rejected': 0.8508, 'Approved': 0.1492}
Row 15 | Not_Graduate, Salaried     , Inc=₹29000, Hist=1, CIBIL=710 | Actual=Approved | Pred=Approved | Probs={'Rejected': 0.2755, 'Approved': 0.7245}

New Loan Predictions:
Applicant 1 | Graduate    , Salaried     , Inc=₹50000, Hist=1, CIBIL=800 | Pred=Approved | Probs={'Rejected': 0.0107, 'Approved': 0.9893}
Applicant 2 | Not_Graduate, Self_Employed, Inc=₹24000, Hist=